In [1]:
#mounting to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#Using shutil to load the data into loacl dir 
import shutil
import os
shutil.copytree('/content/drive/MyDrive/CamVid_data/CamVid', '/content/CamVid')
print(os.listdir('/content/CamVid'))  # verify all folders are there

['val', 'class_dict.csv', 'test', 'val_labels', 'train', 'test_labels', 'train_labels']


In [3]:
import pandas as pd
df = pd.read_csv('/content/CamVid/class_dict.csv')
print(df)

                 name    r    g    b
0              Animal   64  128   64
1             Archway  192    0  128
2           Bicyclist    0  128  192
3              Bridge    0  128   64
4            Building  128    0    0
5                 Car   64    0  128
6     CartLuggagePram   64    0  192
7               Child  192  128   64
8         Column_Pole  192  192  128
9               Fence   64   64  128
10       LaneMkgsDriv  128    0  192
11    LaneMkgsNonDriv  192    0   64
12          Misc_Text  128  128   64
13  MotorcycleScooter  192    0  192
14        OtherMoving  128   64   64
15       ParkingBlock   64  192  128
16         Pedestrian   64   64    0
17               Road  128   64  128
18       RoadShoulder  128  128  192
19           Sidewalk    0    0  192
20         SignSymbol  192  128  128
21                Sky  128  128  128
22     SUVPickupTruck   64  128  192
23        TrafficCone    0    0   64
24       TrafficLight    0   64   64
25              Train  192   64  128
2

In [4]:
#creating a dictionary to map the rgb values to class labels
df = pd.read_csv('/content/CamVid/class_dict.csv')
rgb_to_class={}
for index, rows in df.iterrows():
    rgb = (rows['r'], rows['g'], rows['b'])
    rgb_to_class[rgb] = index
print(rgb_to_class)
print(f"total number of classes: {len(rgb_to_class)}")


{(64, 128, 64): 0, (192, 0, 128): 1, (0, 128, 192): 2, (0, 128, 64): 3, (128, 0, 0): 4, (64, 0, 128): 5, (64, 0, 192): 6, (192, 128, 64): 7, (192, 192, 128): 8, (64, 64, 128): 9, (128, 0, 192): 10, (192, 0, 64): 11, (128, 128, 64): 12, (192, 0, 192): 13, (128, 64, 64): 14, (64, 192, 128): 15, (64, 64, 0): 16, (128, 64, 128): 17, (128, 128, 192): 18, (0, 0, 192): 19, (192, 128, 128): 20, (128, 128, 128): 21, (64, 128, 192): 22, (0, 0, 64): 23, (0, 64, 64): 24, (192, 64, 128): 25, (128, 128, 0): 26, (192, 128, 192): 27, (64, 0, 64): 28, (192, 192, 0): 29, (0, 0, 0): 30, (64, 192, 0): 31}
total number of classes: 32


In [5]:
import numpy as np 
from PIL import Image

def rgbmask_to_class(maskpath):
    mask = np.array(Image.open(maskpath).convert('RGB'))
    class_mask = np.full((mask.shape[0], mask.shape[1]),255,  dtype=np.uint8)
    for rgb, class_label in rgb_to_class.items():
        class_mask[np.all(mask == rgb, axis=-1)] = class_label
    return class_mask


In [6]:
#loading the data 
import os
import torch
from PIL import Image 
from torch.utils.data import Dataset, DataLoader

class Camviddataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir 
        self.images = os.listdir(image_dir)
        self.mask_dir = mask_dir 
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir,self.images[idx].replace('.png', '_L.png'))
        image = np.array(Image.open(image_path).convert('RGB'))
        mask = rgbmask_to_class(mask_path)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        return image, mask.long()

In [7]:
#defining the transformations and using albumentations for data augmentation and normalization , instead of v2 
import albumentations as A
from albumentations.pytorch import ToTensorV2
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Resize(height=512, width=512),
    A.OneOf([
        A.GaussianBlur(p=0.1),
        A.GaussNoise(p=0.1),
    ]),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
]) 

In [8]:
#creating val_transform and test_transform 
import albumentations as A
from albumentations.pytorch import ToTensorV2
test_transform = A.Compose([
    A.Resize(height=512, width=512),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [9]:
#creating datasets
base = '/content/CamVid'
train_dataset=Camviddataset(image_dir=f"{base}/train",
                            mask_dir=f"{base}/train_labels",
                            transform=transform)
test_datasets=Camviddataset(image_dir=f"{base}/test",
                            mask_dir=f"{base}/test_labels",
                            transform=test_transform)
val_datasets=Camviddataset(image_dir=f"{base}/val",
                           mask_dir=f"{base}/val_labels",
                           transform=test_transform)

print(len(train_dataset), len(test_datasets), len(val_datasets))

369 232 100


In [10]:
#turning into dataloader
from torch.utils.data import dataloader

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0, drop_last=True)
test_loader = DataLoader(test_datasets, batch_size=2, shuffle=False, num_workers=0, drop_last=True)
val_loader = DataLoader(val_datasets, batch_size=2,shuffle=False, num_workers=0, drop_last=True)
print(len(train_loader), len(test_loader), len(val_loader))

184 116 50


In [11]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [12]:
#creating Unet from scratch and resnet34 as a backbone
import torch 
import torch.nn as nn
import torchvision.models as models

#Double conv
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
    def forward(self, x):
       return self.conv(x)
    
#BOTTLENECK 
class Bottleneck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x):
        return self.conv(x)

#Decoder
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        upsampled = self.upsample(x)
        concatenated = torch.cat([upsampled, skip], dim=1)
        return self.conv(concatenated)

#Assembling whole Unet class with resnet
from torchvision.models import resnet34, ResNet34_Weights
import torchvision 
import torchvision.models as models
class ResNetUNet(nn.Module):
    def __init__(self,  num_classes:int):
        super().__init__()
        resnet = torchvision.models.resnet34(weights=ResNet34_Weights.DEFAULT, progress=True)
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.enc2 = resnet.layer1
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.enc5 = resnet.layer4

        self.dec1 = DecoderBlock(512, 256, 256)
        self.dec2 = DecoderBlock(256, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.dec4 = DecoderBlock(64, 64, 64)
        self.final_upsample = nn.Upsample(scale_factor=2 ,mode='bilinear', align_corners=False)
        self.output = nn.Conv2d(64, num_classes, kernel_size=1)
    def forward(self, x):
       
       skip1 = self.enc1(x)
       x = self.pool(skip1)
       skip2 = self.enc2(x)
       skip3 = self.enc3(skip2)
       skip4 = self.enc4(skip3)
       x =  self.enc5(skip4)
    
       x = self.dec1(x, skip4)
       x = self.dec2(x, skip3)
       x = self.dec3(x, skip2)
       x = self.dec4(x, skip1)
       
       return self.final_upsample(self.output(x))


In [13]:
model = ResNetUNet(num_classes=32).to(device)  # initialize first
model.load_state_dict(torch.load('/content/drive/MyDrive/Models/ResNetUnet34_best.pth', map_location=device))  # then load weights

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 211MB/s]


<All keys matched successfully>

In [14]:
!pip  install torchinfo
import torchinfo 
from torchinfo import summary
summary(model) 

Layer (type:depth-idx)                   Param #
ResNetUNet                               --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       9,408
│    └─BatchNorm2d: 2-2                  128
│    └─ReLU: 2-3                         --
├─MaxPool2d: 1-2                         --
├─Sequential: 1-3                        --
│    └─BasicBlock: 2-4                   --
│    │    └─Conv2d: 3-1                  36,864
│    │    └─BatchNorm2d: 3-2             128
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  36,864
│    │    └─BatchNorm2d: 3-5             128
│    └─BasicBlock: 2-5                   --
│    │    └─Conv2d: 3-6                  36,864
│    │    └─BatchNorm2d: 3-7             128
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  36,864
│    │    └─BatchNorm2d: 3-10            128
│    └─BasicBlock: 2-6                   --
│    │    └─Conv2d: 3-11                 36,864

In [ ]:
# Dice Loss
import torch.nn as nn 
import torch.nn.functional as F
class DiceLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index

    def forward(self, preds, targets):
        preds = torch.softmax(preds, dim=1)
        
        # handle ignore index FIRST before one_hot
        valid = targets != self.ignore_index
        targets_clean = targets.clone()
        targets_clean[~valid] = 0
        
        # now one_hot on clean targets
        targets_one_hot = torch.nn.functional.one_hot(targets_clean, self.num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()
        
        # mask out ignore regions
        valid = valid.unsqueeze(1).expand_as(preds)
        preds = preds * valid
        targets_one_hot = targets_one_hot * valid
        
        dims = (0, 2, 3)
        intersection = (preds * targets_one_hot).sum(dim=dims)
        union = preds.sum(dim=dims) + targets_one_hot.sum(dim=dims)
        
        return (1 - (2 * intersection) / (union + 1e-6)).mean()
    

In [ ]:
#Combining Dice Loss with Cross Entropy Loss
import torch
class Combined_Loss(nn.Module):
    def __init__(self, num_classes, ignore_index=255):
        super().__init__()
        self.ce = torch.nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.Dice = DiceLoss(num_classes, ignore_index)

    def forward(self, preds, targets):
        return self.ce(preds, targets) + self.Dice(preds, targets)

In [ ]:
#optimizer , loss function and schedular
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-6 )
loss_fn = Combined_Loss(num_classes=32)

In [ ]:
#mIoU
def mIoU(preds, masks, num_classes, ignore_index=255):
    preds = torch.argmax(preds, dim=1)
    iou_list = []
    for cls in range(num_classes):
        valid = masks != ignore_index 
        preds_cls = (preds == cls) & valid 
        masks_cls = (masks == cls) & valid
        intersection = (preds_cls & masks_cls).sum().item()
        union = (preds_cls | masks_cls).sum().item()
        if  union == 0:
            continue

        iou_list.append( intersection / union)
    return sum(iou_list)  / len(iou_list) if iou_list else 0.0


In [ ]:
#training step 
import torch 
import torch.nn as nn
def train_step(model:torch.nn.Module,
               dataloader:torch.utils.data.DataLoader,
               optimizer:torch.optim,
               loss_fn:torch.nn,
               device:torch.device):
    model.train()
    train_loss  = 0 
    for batch, (X, Y) in enumerate(dataloader):
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred , Y)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    print(f" Train loss:{train_loss:.4f} ")
     


In [ ]:
#creating test step 
def test_step(model:torch.nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:torch.nn.Module,
              metric,
              device:torch.device
              ):
    model.eval()
    test_loss, test_acc = 0, 0 
    with torch.inference_mode():
        for batch, (X,Y) in enumerate(dataloader):
            X, Y = X.to(device), Y.to(device) 
            y_preds = model(X)
            loss = loss_fn(y_preds, Y)
            test_loss += loss.item()
            test_acc += metric(preds=y_preds, masks=Y, num_classes=32, ignore_index=255)
        test_loss = test_loss / len(dataloader)
        test_acc = test_acc / len(dataloader)
    print(f" Test loss :{test_loss:.4f} | Test accuracy:{test_acc:.4f}%")


In [21]:
#val step 
def val_step(model:torch.nn.Module,
             dataloader:torch.utils.data.DataLoader,
             loss_fn:torch.nn.Module,
             metric,
             device:torch.device):
    model.eval()
    val_loss, val_acc =0, 0 
    with torch.inference_mode():
        for batch, (X,Y) in enumerate(dataloader):
            X, Y = X.to(device), Y.to(device)
            y_preds = model(X)
            loss = loss_fn(y_preds, Y)
            val_loss += loss.item()
            val_acc += metric(preds=y_preds, masks=Y, num_classes=32, ignore_index=255)
        val_loss = val_loss / len(dataloader)
        val_acc = val_acc / len(dataloader)
        print(f" valulation loss:{val_loss:.4f} | accuarcy:{val_acc:.4f}%")
        return val_loss.item() if isinstance(val_loss, torch.Tensor) else float(val_loss)
        

In [ ]:
#calulating total time 
def total_train_time(start:float, end:float, device:torch.device):
    total_time = end - start 
    min = int( total_time // 60)
    sec = total_time%60
    print(f" Total time spend on device:{device} is {min}minutes and {sec:.2f}seconds")
    return total_time

In [ ]:
#clearing cache
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB allocated")
print(torch.cuda.memory_reserved() / 1024**2, "MB reserved")

95.0419921875 MB allocated
110.0 MB reserved


In [24]:
from tqdm.auto import tqdm
from timeit import default_timer as timer

start_time = timer()
epochs = 15
best_val_loss = float('inf')


for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch+1}/{epochs}\n--")
    train_step(model=model,
               dataloader=train_loader,
               optimizer=optimizer,
               loss_fn=loss_fn,
               device=device)
    test_step(model=model,
               dataloader=test_loader,
               loss_fn=loss_fn,
               metric=mIoU,
               device=device)
    val_loss = val_step(model=model,
               dataloader=val_loader,
               metric=mIoU,
               loss_fn=loss_fn,
               device=device)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 
                   '/content/drive/MyDrive/Models/ResNetUnet34_best.pth')
        print(f"Model saved at epoch {epoch+1}")
    
end_time = timer()
training_time = total_train_time(start=start_time, end=end_time, device=device)
    

  0%|          | 0/15 [00:00<?, ?it/s]

Epoch: 1/15
--
 Train loss:0.9582 
 Test loss :1.1896 | Test accuracy:0.4018%
 valulation loss:1.0451 | accuarcy:0.4412%
Model saved at epoch 1
Epoch: 2/15
--
 Train loss:0.9304 
 Test loss :1.1492 | Test accuracy:0.4298%
 valulation loss:1.0341 | accuarcy:0.4555%
Model saved at epoch 2
Epoch: 3/15
--
 Train loss:0.9208 
 Test loss :1.1121 | Test accuracy:0.4322%
 valulation loss:1.0141 | accuarcy:0.4571%
Model saved at epoch 3
Epoch: 4/15
--
 Train loss:0.9193 
 Test loss :1.1099 | Test accuracy:0.4407%
 valulation loss:0.9992 | accuarcy:0.4691%
Model saved at epoch 4
Epoch: 5/15
--
 Train loss:0.8944 
 Test loss :1.0896 | Test accuracy:0.4477%
 valulation loss:0.9932 | accuarcy:0.4768%
Model saved at epoch 5
Epoch: 6/15
--
 Train loss:0.8889 
 Test loss :1.1195 | Test accuracy:0.4386%
 valulation loss:0.9926 | accuarcy:0.4693%
Model saved at epoch 6
Epoch: 7/15
--
 Train loss:0.9001 
 Test loss :1.1715 | Test accuracy:0.4331%
 valulation loss:1.0216 | accuarcy:0.4690%
Epoch: 8/15
--
